# Can we predict 30-day readmission?
Target: `readmit_30d` (data.csv, dictionary in columns.csv).

Key design point: the 11.9k rows are **repeated hospital stays of only 1,500 people** (~8 stays each). Rows from one person share traits, so a random row-wise split would leak person identity between train and test and overstate performance. All evaluation here is **grouped by `person_key`**; a random-split run is included only to show the size of the leakage.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 200)
df = pd.read_csv('data.csv'); print(df.shape); df.head()

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(), 'missing;', df.duplicated().sum(), 'duplicate rows;', df.person_key.nunique(), 'people')
print('Base rate:', df.readmit_30d.mean().round(4))
print(df.groupby('person_key').nunique().max().rename('max distinct values within a person'))
df.describe().T

In [ ]:
print(df.groupby('visit_number').readmit_30d.agg(['mean', 'size']).T.round(3))

## 2. Univariate signal
Readmission rate by octile of each numeric feature.

In [ ]:
num = ['birth_year', 'bmi', 'length_of_stay_days', 'n_medications', 'lab_a', 'lab_b', 'postcode_area', 'visit_number']
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, col in zip(ax.ravel(), num):
    q = pd.qcut(df[col], 8, duplicates='drop')
    r = df.groupby(q, observed=True).readmit_30d.mean()
    a.plot(range(len(r)), r.values, 'o-'); a.axhline(df.readmit_30d.mean(), ls='--', c='gray')
    a.set_title(col); a.set_xlabel('octile'); a.set_ylim(0, .5)
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())
print(df[num + ['readmit_30d']].corr()['readmit_30d'].round(3))

## 3. Models, evaluated with person-grouped 5-fold CV
Baselines: constant prior, logistic regression, gradient boosting. `person_key` is **not** a feature.

In [ ]:
X = df.drop(columns=['readmit_30d', 'person_key']); X['female'] = (X.pop('sex') == 'F').astype(int)
y = df.readmit_30d.values; g = df.person_key.values
models = {
    'prior (dummy)': DummyClassifier(strategy='prior'),
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'hist-gbm': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=0),
}
def evaluate(cv, groups=None):
    rows, preds = {}, {}
    for n, m in models.items():
        p = cross_val_predict(m, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]; preds[n] = p
        rows[n] = dict(AUC=roc_auc_score(y, p), AP=average_precision_score(y, p), Brier=brier_score_loss(y, p))
    return pd.DataFrame(rows).T.round(4), preds
res_g, pred_g = evaluate(GroupKFold(5), g); print('GROUPED CV (honest)'); display(res_g)
res_r, _ = evaluate(StratifiedKFold(5, shuffle=True, random_state=0)); print('RANDOM row-wise CV (leaky, for comparison)'); display(res_r)

### Uncertainty: bootstrap over people

In [ ]:
rng = np.random.default_rng(0); ids = np.unique(g); idx = {i: np.where(g == i)[0] for i in ids}
def boot(p, B=300):
    out = []
    for _ in range(B):
        s = np.concatenate([idx[i] for i in rng.choice(ids, len(ids))]); out.append(roc_auc_score(y[s], p[s]))
    return np.percentile(out, [2.5, 50, 97.5]).round(4)
for n in ['logistic', 'hist-gbm']: print(n, 'AUC 95% CI (lo, med, hi):', boot(pred_g[n]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for n, p in pred_g.items():
    f, t, _ = roc_curve(y, p); ax[0].plot(f, t, label=f'{n} ({roc_auc_score(y, p):.3f})')
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend(); ax[0].set_title('ROC (grouped CV)')
for n in ['logistic', 'hist-gbm']:
    fo, mp = calibration_curve(y, pred_g[n], n_bins=10, strategy='quantile'); ax[1].plot(mp, fo, 'o-', label=n)
ax[1].plot([0, .4], [0, .4], 'k:'); ax[1].set_xlabel('predicted'); ax[1].set_ylabel('observed'); ax[1].set_title('Calibration'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. What drives the prediction? (permutation importance, held-out people)

In [ ]:
tr, te = next(GroupShuffleSplit(1, test_size=.25, random_state=1).split(X, y, g))
m = models['logistic'].fit(X.iloc[tr], y[tr]); print('holdout AUC logistic:', round(roc_auc_score(y[te], m.predict_proba(X.iloc[te])[:, 1]), 4))
pi = permutation_importance(m, X.iloc[te], y[te], scoring='roc_auc', n_repeats=20, random_state=0)
imp = pd.Series(pi.importances_mean, X.columns).sort_values()
imp.plot.barh(xerr=pd.Series(pi.importances_std, X.columns)[imp.index]); plt.xlabel('AUC drop'); plt.show()
print(pd.Series(m[-1].coef_[0], X.columns).sort_values().round(3))

## 5. Practical usefulness: lift by predicted-risk decile

In [ ]:
p = pred_g['logistic']; d = pd.DataFrame({'y': y, 'p': p}); d['decile'] = pd.qcut(d.p.rank(method='first'), 10, labels=False) + 1
display(d.groupby('decile').agg(n=('y', 'size'), observed=('y', 'mean'), mean_pred=('p', 'mean')).round(3))
top = d[d.decile == 10]; print('Top-decile rate %.3f vs base %.3f (lift %.2fx)' % (top.y.mean(), y.mean(), top.y.mean() / y.mean()))

## 6. Bottom line

In [ ]:
best = res_g.loc[['logistic', 'hist-gbm'], 'AUC'].idxmax()
print(f"Best grouped-CV AUC: {best} = {res_g.loc[best, 'AUC']:.3f} (chance = 0.5). Random-split AUC would have said {res_r.loc[best, 'AUC']:.3f}.")